# Cox Regression Penalized

### Imports

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

# Clinical Data
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

# TODO : Uncomment for test data ??
"""
target_df_test = pd.read_csv("./data/target_test.csv")
"""
# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


### Cleaning

In [136]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

OS_STATUS    float64
OS_YEARS     float64
dtype: object


0. Parsing functions

In [543]:
def parse_protein(protein):
    protein = str(protein)

    if len(protein) == 0 or len(protein) == 1:
        return pd.NA
    
    try:
        protein = str(protein.split('.')[1])
    
    except IndexError:
        return pd.NA
    
    if protein == '?' or protein[0] == '*':
        return pd.NA
    
    else:
        return protein[0]
    
def parse_ref_alt(ref):
    ref = str(ref)
    if len(ref) == 0:
        return pd.NA
    elif ref[0] == '-':
        return pd.NA
    else:
        return ref[0]
    
def parse_chromosome(chr):
    chr = str(chr)
    if chr not in ['1','2','3','4','5','6','7','8','9','10',
                   '11','12','13','14','15','16','17','18',
                   '19','20','21','22','X','Y']:
        return pd.NA
    else:
        return chr
    
def parse_gene(gene):
    gene = str(gene)
    if len(gene) == 0:
        return pd.NA
    elif len(gene) < 4:
        return gene
    else:
        return gene[0:4]
    
def parse_cytogenetics(karyotype):
    karyotype = str(karyotype)
    results = []
    segments = karyotype.split("/")  # sépare 46,xx[...] / 45,xx[...]

    for seg in segments:
        anomalies = re.findall(r"(del\(\d+\)|add\(\d+\)|\+\d+|-\d+)", seg)
        results.extend(anomalies)

    return results

1. Extracting number of somatic mutations and cytogenetics parsed

In [544]:
# Step: Extract the number of somatic mutations per patient
# Group by 'ID' and count the number of mutations (rows) per patient
tmp = maf_df.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_2 = df.merge(tmp, on='ID', how='left').fillna({'Nmut': 0})

df_2['CYTOGENETICS_ABNORMALITIES'] = df_2['CYTOGENETICS'].apply(parse_cytogenetics)

from sklearn.preprocessing import MultiLabelBinarizer

# Initialize the MultiLabelBinarizer
mlb = MultiLabelBinarizer()

# One-hot encode the CYTOGENETICS_ABNORMALITIES column
cytogenetics_encoded = mlb.fit_transform(df_2['CYTOGENETICS_ABNORMALITIES'])

# Create a DataFrame for the encoded features
cytogenetics_df = pd.DataFrame(cytogenetics_encoded, columns=mlb.classes_, index=df_2.index)

# Merge the one-hot encoded features back into the original DataFrame
df_2 = pd.concat([df_2, cytogenetics_df], axis=1)
df_2.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,Nmut,...,del(20),del(21),del(22),del(3),del(4),del(5),del(6),del(7),del(8),del(9)
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",9.0,...,1,0,0,0,0,0,0,0,0,0
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx",3.0,...,0,0,0,0,0,0,0,0,0,0
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",3.0,...,0,0,0,0,0,0,0,0,0,0
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",11.0,...,0,0,0,1,0,0,0,0,0,0
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",1.0,...,0,0,0,0,0,0,0,0,0,0


2. One hot encoding 'GENE', 'EFFECT' and 2 first relevant characters of 'PROTEIN_CHANGE'

In [558]:
# One-hot encode the 'GENE', 'EFFECT', 'REF', 'PROTEIN_CHANGE' columns
one_hot_encoded_genes = pd.get_dummies(maf_df['GENE'].apply(parse_gene), prefix='GENE')
one_hot_encoded_effects = pd.get_dummies(maf_df['EFFECT'], prefix='EFFECT')
one_hot_encoded_proteins = pd.get_dummies(maf_df['PROTEIN_CHANGE'].apply(parse_protein), prefix='PROTEIN')
one_hot_encoded_refs = pd.get_dummies(maf_df['REF'].apply(parse_ref_alt),prefix='REF')
one_hot_encoded_alts = pd.get_dummies(maf_df['ALT'].apply(parse_ref_alt),prefix='ALT')
one_hot_encoded_chromosomes = pd.get_dummies(maf_df['CHR'].apply(parse_chromosome),prefix='CHR')

# Add the one-hot encoded columns to the original DataFrame
maf_df_one_hot_genes = pd.concat([maf_df[['ID']], one_hot_encoded_genes], axis=1)
maf_df_one_hot_effects = pd.concat([maf_df[['ID']], one_hot_encoded_effects], axis=1)
maf_df_one_hot_proteins = pd.concat([maf_df[['ID']], one_hot_encoded_proteins], axis=1)
maf_df_one_hot_refs = pd.concat([maf_df['ID'], one_hot_encoded_refs],axis=1)
maf_df_one_hot_alts = pd.concat([maf_df['ID'], one_hot_encoded_alts],axis=1)
maf_df_one_hot_chromosomes = pd.concat([maf_df['ID'], one_hot_encoded_chromosomes],axis=1)

# Group by 'ID' and aggregate using max to ensure one entry per ID
maf_df_one_hot_genes = maf_df_one_hot_genes.groupby('ID').max().reset_index()
maf_df_one_hot_effects = maf_df_one_hot_effects.groupby('ID').max().reset_index()
maf_df_one_hot_proteins = maf_df_one_hot_proteins.groupby('ID').max().reset_index()
maf_df_one_hot_refs = maf_df_one_hot_refs.groupby('ID').max().reset_index()
maf_df_one_hot_alts = maf_df_one_hot_alts.groupby('ID').max().reset_index()
maf_df_one_hot_chromosomes = maf_df_one_hot_chromosomes.groupby('ID').max().reset_index()

# Merge all one-hot encoded DataFrames into a single grouped DataFrame
maf_df_one_hot_grouped = (
    maf_df_one_hot_proteins
    .merge(maf_df_one_hot_genes, on='ID', how='outer')
    .merge(maf_df_one_hot_effects, on='ID', how='outer')
    .merge(maf_df_one_hot_refs, on='ID', how='outer')
    .merge(maf_df_one_hot_alts, on='ID', how='outer')
    .merge(maf_df_one_hot_chromosomes, on='ID', how='outer')
    .fillna(0)
)
maf_df_one_hot_grouped.head()

,ID,PROTEIN_A,PROTEIN_C,PROTEIN_D,PROTEIN_E,PROTEIN_F,PROTEIN_G,PROTEIN_H,PROTEIN_I,PROTEIN_K,...,CHR_21,CHR_22,CHR_3,CHR_4,CHR_5,CHR_6,CHR_7,CHR_8,CHR_9,CHR_X
0,P100000,False,True,False,True,False,False,False,False,False,...,False,False,True,True,True,False,False,False,False,False
1,P100001,False,False,False,True,False,False,False,False,False,...,False,True,True,False,False,False,False,False,False,False
2,P100002,False,False,False,True,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,P100004,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,P100006,False,False,False,False,False,False,False,False,False,...,False,True,False,True,False,False,False,False,True,False


### 3. Training

1. Selecting features and getting rid of rows with NaN values in training set

In [580]:
# Select features and target columns
features = ['BM_BLAST', 'HB', 'PLT', 'Nmut', 'ANC','WBC']
one_hot_features_genes = list(maf_df_one_hot_genes.columns[1:])
one_hot_features_effects = list(maf_df_one_hot_effects.columns[1:])
one_hot_encoded_proteins = list(maf_df_one_hot_proteins.columns[1:])
one_hot_encoded_refs = list(maf_df_one_hot_refs.columns[1:])
one_hot_encoded_alts = list(maf_df_one_hot_alts.columns[1:])
one_hot_encoded_chromosomes = list(maf_df_one_hot_chromosomes.columns[1:])
one_hot_encoded_cytogenetics = list(cytogenetics_df.columns)
# Final feature list
total_features = features + one_hot_features_effects + one_hot_features_genes + one_hot_encoded_proteins + one_hot_encoded_refs + one_hot_encoded_alts + one_hot_encoded_chromosomes + one_hot_encoded_cytogenetics

target = ['OS_YEARS', 'OS_STATUS']

# Merge on ID to ensure same row order
merged = df_2[['ID','CENTER'] + features].merge(target_df[['ID'] + target], on='ID', how='left')
merged = merged.merge(maf_df_one_hot_grouped, on='ID', how='left')  # Merge one-hot encoded genes
merged = merged.merge(cytogenetics_df, left_index=True, right_index=True, how='left')  # Merge one-hot encoded cytogenetics
# Remove rows with missing feature values
merged = merged.dropna()

# Build feature matrix and survival target
X = merged[total_features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', merged[target])

2. Spliting dataset

In [574]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
X_train.head()

,BM_BLAST,HB,PLT,Nmut,ANC,WBC,EFFECT_2KB_upstream_variant,EFFECT_3_prime_UTR_variant,EFFECT_ITD,EFFECT_PTD,...,del(20),del(21),del(22),del(3),del(4),del(5),del(6),del(7),del(8),del(9)
1793,5.0,10.600,175.0,5.0,2.80,3.8,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
3043,3.0,9.100,247.0,2.0,1.90,4.9,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
2920,1.0,6.927,274.0,3.0,2.00,4.2,False,False,False,False,...,0,0,0,0,1,1,0,0,0,0
918,2.0,6.600,51.0,5.0,5.06,8.3,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
2923,18.0,9.022,103.0,6.0,0.70,1.9,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0


3. Fitting Cox Ridge model

In [575]:
# Initialize and train the Cox Proportional Hazards model
cox = CoxnetSurvivalAnalysis(l1_ratio=1, alpha_min_ratio=0.1, max_iter=1000, normalize=True)
cox.fit(X_train, y_train)

# Evaluate the model using Concordance Index IPC
cox_cindex_train = concordance_index_ipcw(y_train, y_train, cox.predict(X_train), tau=7)[0]
cox_cindex_test = concordance_index_ipcw(y_train, y_test, cox.predict(X_test), tau=7)[0]
print(f"Cox Proportional Hazard Model Concordance Index IPCW on train: {cox_cindex_train:.5f}")
print(f"Cox Proportional Hazard Model Concordance Index IPCW on test: {cox_cindex_test:.5f}")

Cox Proportional Hazard Model Concordance Index IPCW on train: 0.73598
Cox Proportional Hazard Model Concordance Index IPCW on test: 0.71881


### 4. Prediction

1. Calculating Nmut on prediction sample

In [549]:
tmp_eval = maf_eval.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_eval2 = df_eval.merge(tmp_eval, on='ID', how='left').fillna({'Nmut': 0})

2. One hot encoding 'GENE', 'EFFECT', 'PROTEINS'

In [581]:
# One-hot encode the 'GENE' and 'EFFECT' column
one_hot_encoded_genes_eval = pd.get_dummies(maf_eval['GENE'], prefix='GENE')
one_hot_encoded_effects_eval = pd.get_dummies(maf_eval['EFFECT'], prefix='EFFECT')
one_hot_encoded_proteins_eval = pd.get_dummies(maf_eval['PROTEIN_CHANGE'].apply(parse_protein), prefix='PROTEIN')

# Add the one-hot encoded columns to the original DataFrame
maf_eval_one_hot_genes = pd.concat([maf_eval[['ID']], one_hot_encoded_genes_eval], axis=1)
maf_eval_one_hot_effects = pd.concat([maf_eval[['ID']], one_hot_encoded_effects_eval], axis=1)
maf_eval_one_hot_proteins = pd.concat([maf_eval[['ID']], one_hot_encoded_proteins_eval], axis=1)

# Group by 'ID' and aggregate using max to ensure one entry per ID
maf_eval_one_hot_grouped = maf_eval_one_hot_proteins.merge(maf_eval_one_hot_genes.merge(maf_eval_one_hot_effects, on='ID', how='outer'), on='ID', how='outer').fillna(0)
maf_eval_one_hot_grouped = maf_eval_one_hot_grouped.groupby('ID').max().reset_index()

maf_eval_one_hot_grouped.head()

,ID,PROTEIN_1,PROTEIN_2,PROTEIN_3,PROTEIN_4,PROTEIN_5,PROTEIN_6,PROTEIN_7,PROTEIN_8,PROTEIN_9,...,GENE_WT1,GENE_ZRSR2,EFFECT_ITD,EFFECT_PTD,EFFECT_frameshift_variant,EFFECT_inframe_codon_gain,EFFECT_inframe_codon_loss,EFFECT_non_synonymous_codon,EFFECT_stop_gained,EFFECT_stop_lost
0,KYW1,False,False,False,False,False,False,False,False,False,...,False,False,True,False,True,False,False,True,True,False
1,KYW10,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,True,True,False
2,KYW100,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
3,KYW1000,False,False,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,False,True,False
4,KYW1001,False,False,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,True,True,False


3. Align colums of X_eval with those of X_train

In [584]:
X_eval = df_eval2[['ID'] + features].merge(maf_eval_one_hot_grouped, on='ID', how='left')
X_eval = X_eval.drop(columns=['ID'])

# Ensure X is a pandas DataFrame before accessing its columns
if not isinstance(X, pd.DataFrame):
	X = pd.DataFrame(X, columns=total_features)

X_eval = X_eval.reindex(columns=X.columns, fill_value=False)
X_eval = X_eval.drop(columns=[col for col in X_eval.columns if col not in total_features])
X_eval.head()

,BM_BLAST,HB,PLT,Nmut,ANC,WBC,EFFECT_2KB_upstream_variant,EFFECT_3_prime_UTR_variant,EFFECT_ITD,EFFECT_PTD,...,del(20),del(21),del(22),del(3),del(4),del(5),del(6),del(7),del(8),del(9)
0,68.0,7.6,48.0,4.0,0.5865,3.45,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1,35.0,10.0,32.0,3.0,1.2402,3.18,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,NaN,12.3,25.0,3.0,8.6800,12.40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,61.0,8.0,44.0,3.0,2.0535,5.55,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,2.0,8.6,27.0,3.0,0.7381,1.21,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


3. Imputer with median strategy

In [586]:
imputer = SimpleImputer(strategy='median')
print(features)
X_train[features] = imputer.fit_transform(X_train[features])
X_eval[features] = imputer.transform(X_eval[features])
X_eval = X_eval.fillna(False)
X_eval.head()

['BM_BLAST', 'HB', 'PLT', 'Nmut', 'ANC', 'WBC']


,BM_BLAST,HB,PLT,Nmut,ANC,WBC,EFFECT_2KB_upstream_variant,EFFECT_3_prime_UTR_variant,EFFECT_ITD,EFFECT_PTD,...,del(20),del(21),del(22),del(3),del(4),del(5),del(6),del(7),del(8),del(9)
0,68.0,7.6,48.0,4.0,0.5865,3.45,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1,35.0,10.0,32.0,3.0,1.2402,3.18,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,3.5,12.3,25.0,3.0,8.6800,12.40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,61.0,8.0,44.0,3.0,2.0535,5.55,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,2.0,8.6,27.0,3.0,0.7381,1.21,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [587]:
prediction_on_test_set = cox.predict(X_eval)
prediction_on_test_set

array([ 1.53789231,  0.28938291, -0.57668405, ..., -0.1450719 ,
       -0.44887364, -0.15646978])

In [588]:
submission = pd.Series(prediction_on_test_set, index=df_eval['ID'], name='risk_score')

In [589]:
submission.to_csv('./submission/cox_ridge_submission.csv')

In [590]:
submission.head()

ID
KYW1    1.537892
KYW2    0.289383
KYW3   -0.576684
KYW4    1.284838
KYW5   -0.220335
Name: risk_score, dtype: float64